# 패션 이커머스 AI 어시스턴트 — 03. 의미 검색 (Semantic Search)

**02에서 확인한 한계**
- 정답 상품의 19.5%가 학습 데이터에 없어서 협업 필터링으로는 절대 추천할 수 없었다 (아이템 콜드 스타트)
- 카시오 시계와 "비슷한 상품"에 아동 블레이저, 양말이 나왔다. 구매 기록만으로는 상품이 **무엇인지** 모른다

**이번 노트북에서 할 일**
1. 상품마다 검색용 문서 만들기 — v1: 상품명+특징+설명 / v2: v1 + 대표 리뷰
2. 다국어 임베딩 모델로 상품을 벡터로 바꾸고 FAISS로 검색 인덱스 만들기
3. 키워드 검색(BM25)과 비교하고, 둘을 합친 **하이브리드 검색** 만들기
4. 한국어 질문으로 영어 상품 검색해보기
5. **콘텐츠 기반 추천으로 콜드 스타트 상품을 얼마나 맞히는지** 측정 (02의 CF 모델은 0%)
6. 02의 카시오 시계 '비슷한 상품'과 비교

> EDA 정리 질문 4번에서 세운 계획("설명이 92.8% 비어 있으니 리뷰를 섞는다", "키워드 검색과 결합한다")을 실제로 검증하는 노트북이에요.

## 0. 환경 설정
GPU 런타임을 권장해요 (런타임 > 런타임 유형 변경 > GPU). 임베딩 계산이 CPU보다 수십 배 빨라요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/fashion-ai-assistant'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
MODEL_DIR = f'{PROJECT_DIR}/models'
EMB_DIR = f'{PROJECT_DIR}/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)
K = 2

Mounted at /content/drive


In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25

import re
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.5 MB/s eta 0:00:00


## 1. 데이터 불러오기
- `items`: 01에서 저장한 상품 정보 (검색 대상)
- `reviews`: 전체 리뷰 (검색 문서에 섞을 재료)
- `interactions`: 5번 콜드 스타트 평가에 사용. 02와 **똑같은 방법으로** 학습/평가를 나눠요

In [ ]:
items = pd.read_parquet(f'{PROCESSED_DIR}/items_k{K}.parquet').reset_index(drop=True)
inter = pd.read_parquet(f'{PROCESSED_DIR}/interactions_k{K}.parquet')

# 02와 동일한 Leave-One-Out 분할
inter = inter.sort_values(['user_id', 'timestamp'])
is_last = inter.groupby('user_id').cumcount(ascending=False) == 0
train_df = inter[~is_last].copy()
test_df = inter[is_last].copy()
test_df['is_cold'] = ~test_df['parent_asin'].isin(set(train_df['parent_asin']))

print(f'상품: {len(items):,} | train: {len(train_df):,} | test: {len(test_df):,}')
print(f"정답이 콜드 스타트 상품인 비율: {test_df['is_cold'].mean():.1%}  (02의 19.5%와 같아야 해요)")

상품: 70,641 | train: 171,028 | test: 118,869
정답이 콜드 스타트 상품인 비율: 19.5%  (02의 19.5%와 같아야 해요)


In [ ]:
reviews = pd.read_parquet(f'{PROCESSED_DIR}/reviews_all.parquet',
                          columns=['user_id', 'parent_asin', 'rating', 'text', 'helpful_vote'])
reviews = reviews[reviews['parent_asin'].isin(set(items['parent_asin']))]
print(f'검색 대상 상품의 리뷰: {len(reviews):,}')

검색 대상 상품의 리뷰: 1,112,805


## 2. 검색용 문서 만들기

**v1**: 상품명 + 특징 + 설명 (판매자가 쓴 정보)
**v2**: v1 + 도움돼요가 많은 리뷰 최대 3개 (구매자가 쓴 정보)

⚠️ **데이터 누수 방지**: 5번에서 평가할 정답 리뷰(test 유저가 정답 상품에 남긴 리뷰)는 문서에서 뺍니다.
그대로 두면 "정답을 쓴 사람의 문장"이 검색 문서에 들어가서 점수가 부풀려져요.

In [ ]:
N_REVIEWS = 3
MAX_REVIEW_CHARS = 300

def clean(s):
    return re.sub(r'\s+', ' ', str(s or '')).strip()

items['doc_v1'] = (items['title'].fillna('') + '. ' +
                   items['features'].fillna('') + ' ' +
                   items['description'].fillna('')).map(clean)

# 평가 정답 리뷰 제외
test_keys = pd.MultiIndex.from_frame(test_df[['user_id', 'parent_asin']])
rev = reviews[~pd.MultiIndex.from_frame(reviews[['user_id', 'parent_asin']]).isin(test_keys)].copy()

rev['text'] = rev['text'].fillna('').map(clean).str.slice(0, MAX_REVIEW_CHARS)
rev = rev[rev['text'].str.len() >= 20]   # "Love it!" 같은 짧은 리뷰는 정보가 적어서 제외

top_rev = (rev.sort_values(['parent_asin', 'helpful_vote'], ascending=[True, False])
              .groupby('parent_asin').head(N_REVIEWS))
snippets = top_rev.groupby('parent_asin')['text'].agg(' | '.join)

items['review_snippets'] = items['parent_asin'].map(snippets).fillna('')
items['doc_v2'] = np.where(items['review_snippets'] != '',
                           items['doc_v1'] + ' Reviews: ' + items['review_snippets'],
                           items['doc_v1'])

wc = lambda s: s.str.split().str.len()
print(f"리뷰가 붙은 상품 비율: {(items['review_snippets'] != '').mean():.1%}")
print(f"문서 길이 중앙값(단어): v1 {wc(items['doc_v1']).median():.0f} → v2 {wc(items['doc_v2']).median():.0f}")
items[['title', 'doc_v2']].sample(3, random_state=1)

리뷰가 붙은 상품 비율: 96.8%
문서 길이 중앙값(단어): v1 17 → v2 104


,title,doc_v2
28708,Red Dust Active Sports Headbands – Breathable Bamboo Exe...,Red Dust Active Sports Headbands – Breathable Bamboo Exe...
43457,Jescakoo Womens Tank Dresses Sleeveless Pleated Skater S...,Jescakoo Womens Tank Dresses Sleeveless Pleated Skater S...
16804,Lares Domi Bohemian Gold-tone Simulated Gemstones and Pe...,Lares Domi Bohemian Gold-tone Simulated Gemstones and Pe...


## 3. 임베딩 모델

`intfloat/multilingual-e5-base`를 써요.
- **다국어 모델**이라 "여름 원피스"(한국어)와 "summer dress"(영어)가 비슷한 벡터가 돼요. 영어 상품을 한국어로 검색할 수 있는 이유예요.
- e5 모델은 앞에 **`query: `**, **`passage: `** 를 붙여야 성능이 제대로 나와요 (모델 설명서의 규칙).
- 벡터를 길이 1로 정규화하면 **내적 = 코사인 유사도**가 돼요.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 256          # 리뷰가 붙은 문서도 대부분 들어가는 길이
if device == 'cuda':
    model.half()                     # GPU에서 메모리 절약 + 속도 향상
print('device:', device)

def embed_passages(texts, batch_size=128):
    return model.encode(['passage: ' + t for t in texts], batch_size=batch_size,
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=True).astype(np.float32)

def embed_queries(texts):
    return model.encode(['query: ' + t for t in texts],
                        normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

device: cuda


임베딩은 시간이 걸리니 드라이브에 저장해 두고, 다음에는 불러오기만 해요. 문서 내용을 바꿨다면 파일을 지우고 다시 계산하세요.

In [ ]:
def load_or_embed(name, texts):
    path = f'{EMB_DIR}/{name}.npy'
    if os.path.exists(path):
        emb = np.load(path).astype(np.float32)
        if len(emb) == len(texts):
            print(f'{name}: 저장된 임베딩 불러옴 {emb.shape}')
            return emb
    emb = embed_passages(texts.tolist())
    np.save(path, emb.astype(np.float16))   # 용량 절반
    print(f'{name}: 계산 후 저장 {emb.shape}')
    return emb

emb = {
    'v1': load_or_embed('items_v1', items['doc_v1']),
    'v2': load_or_embed('items_v2', items['doc_v2']),
}

Batches:   0%|          | 0/552 [00:00<?, ?it/s]

items_v1: 계산 후 저장 (70641, 768)


Batches:   0%|          | 0/552 [00:00<?, ?it/s]

items_v2: 계산 후 저장 (70641, 768)


## 4. 검색 인덱스

### 4-1. FAISS (벡터 검색)
`IndexFlatIP`는 모든 상품과 내적을 계산하는 가장 단순한 인덱스예요. 7만 개 정도는 이걸로 충분히 빨라요.
상품이 수천만 개가 되면 IVF, HNSW 같은 **근사 검색** 인덱스로 바꿉니다.

In [ ]:
import faiss

index = {}
for v, e in emb.items():
    index[v] = faiss.IndexFlatIP(e.shape[1])
    index[v].add(e)
    print(v, index[v].ntotal)

v1 70641
v2 70641


### 4-2. BM25 (키워드 검색)
검색어 단어가 문서에 **정확히** 몇 번 나오는지로 점수를 매기는 전통적인 방법이에요.
브랜드명, 모델명(예: "F91W")처럼 정확히 일치해야 하는 검색어에 강하고, 한국어 검색어는 영어 문서와 단어가 안 겹쳐서 아예 못 찾아요.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r'[a-z0-9]+', s.lower())

bm25 = BM25Okapi([tokenize(t) for t in items['doc_v1']])
print('BM25 인덱스 완료')

BM25 인덱스 완료


## 5. 검색 함수

**하이브리드 검색 (RRF, Reciprocal Rank Fusion)**: 두 검색 결과의 **순위**만 보고 합쳐요.
`점수 = 1/(60 + 벡터검색 순위) + 1/(60 + BM25 순위)`
점수 단위가 다른 두 방법(코사인 유사도 vs BM25 점수)을 정규화 없이 합칠 수 있어서 실무에서 많이 써요.

In [ ]:
def search_dense(query, version='v2', k=10):
    scores, ids = index[version].search(embed_queries([query]), k)
    return list(zip(ids[0], scores[0]))

def search_bm25(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(i, scores[i]) for i in top if scores[i] > 0]

def search_hybrid(query, k=10, pool=50, rrf_k=60):
    fused = {}
    for results in (search_dense(query, 'v2', pool), search_bm25(query, pool)):
        for rank, (i, _) in enumerate(results):
            fused[i] = fused.get(i, 0) + 1 / (rrf_k + rank + 1)
    top = sorted(fused.items(), key=lambda x: -x[1])[:k]
    return top

METHODS = {
    'dense_v1': lambda q, k: search_dense(q, 'v1', k),
    'dense_v2': lambda q, k: search_dense(q, 'v2', k),
    'bm25':     search_bm25,
    'hybrid':   search_hybrid,
}

In [ ]:
def show_items(rows, title, scores=None):
    cards = []
    for n, r in enumerate(rows):
        it = items.iloc[r]
        name = (it['title'] or it['parent_asin'])[:60]
        img = it['image_url']
        img_tag = (f'<img src="{img}" style="height:110px;object-fit:contain">' if img
                   else '<div style="height:110px">no image</div>')
        sc = f'<br><b>{scores[n]:.3f}</b>' if scores is not None else ''
        cards.append(f'<div style="width:130px;font-size:11px;text-align:center">{img_tag}<br>{name}{sc}</div>')
    display(HTML(f'<h4>{title}</h4><div style="display:flex;gap:8px;flex-wrap:wrap">{"".join(cards)}</div>'))

def show_search(query, k=8):
    display(HTML(f'<h3>🔎 {query}</h3>'))
    for name, fn in METHODS.items():
        res = fn(query, k)
        show_items([i for i, _ in res], name, [s for _, s in res])

show_search('여름에 입기 좋은 시원한 린넨 원피스')

## 6. 검색 방법 비교

한국어 / 영어, 분위기 표현 / 정확한 이름 검색어를 섞었어요. 직접 검색어를 추가해 보세요.
이미지로 보기 무거우니 상위 5개 상품명만 표로 비교합니다.

In [ ]:
QUERIES = [
    '여름에 입기 좋은 시원한 린넨 원피스',
    '결혼식 하객룩 원피스',
    '출근할 때 입을 단정한 블라우스',
    '운동할 때 입는 하이웨이스트 레깅스',
    '실버 원석 반지',
    'cozy pajamas for winter',
    'black leather crossbody bag',
    'Casio F91W',
    'Nike running shoes',
    '작게 나와서 한 치수 크게 사야 하는 청바지',
]

def compare_table(query, k=5):
    cols = {}
    for name, fn in METHODS.items():
        res = fn(query, k)
        titles = [items.iloc[i]['title'][:50] for i, _ in res]
        cols[name] = titles + [''] * (k - len(titles))
    return pd.DataFrame(cols, index=range(1, k + 1))

for q in QUERIES:
    print(f'\n🔎 {q}')
    display(compare_table(q))


🔎 여름에 입기 좋은 시원한 린넨 원피스


,dense_v1,dense_v2,bm25,hybrid
1,Womens Linen Overalls Shorts Summer Casual Rompers,Yuandy Womens Summer Casual Off Shoulder Print Jum,,Yuandy Womens Summer Casual Off Shoulder Print Jum
2,Yuandy Womens Summer Casual Off Shoulder Print Jum,Mnemo Women's Summer Jumpsuits Sleeveless Casual P,,Mnemo Women's Summer Jumpsuits Sleeveless Casual P
3,Soojun Women's Solid Cotton Linen Pants Summer Wid,Womens Linen Overalls Shorts Summer Casual Rompers,,Womens Linen Overalls Shorts Summer Casual Rompers
4,Womens Linen Overalls Shorts Summer Casual Rompers,Ingear Summer Jumper Long Tie Dye Strapless Beach,,Ingear Summer Jumper Long Tie Dye Strapless Beach
5,Womens Linen Overalls Shorts Summer Casual Rompers,REORIA Womens Casual Summer One Piece Ruffles Shor,,REORIA Womens Casual Summer One Piece Ruffles Shor



🔎 결혼식 하객룩 원피스


,dense_v1,dense_v2,bm25,hybrid
1,Fanhao Hoop Skirt Wedding Slip Bridal Petticoat/Un,Mojessy Women's Strapless Stripe High Waist Wide L,,Mojessy Women's Strapless Stripe High Waist Wide L
2,Wedding Mermaid Dovetail White Prom Dress Petticoa,Hotouch Sequin Bodice with Double Layer Mesh Flowe,,Hotouch Sequin Bodice with Double Layer Mesh Flowe
3,INBLUE Men's Rhodium Plated Cufflinks Silver Tone,Fanhao Hoop Skirt Wedding Slip Bridal Petticoat/Un,,Fanhao Hoop Skirt Wedding Slip Bridal Petticoat/Un
4,"Cocktail Dress,Zalalus Women's Chic High Neck Prom",Women Wedding Robes Half Sleeve Mid Thigh Dressing,,Women Wedding Robes Half Sleeve Mid Thigh Dressing
5,HaoDuoYi Womens Sequin V Neck Wrap Tunic Party Jum,YING LAN Girl Round-Toe Sparkle Bowknot Ballet Bal,,YING LAN Girl Round-Toe Sparkle Bowknot Ballet Bal



🔎 출근할 때 입을 단정한 블라우스


...

,dense_v1,dense_v2,bm25,hybrid
1,Casio F91W-1 Classic Resin Strap Digital Sport Wat,Casio F91W-1 Classic Resin Strap Digital Sport Wat,Casio F91W-1 Classic Resin Strap Digital Sport Wat,Casio F91W-1 Classic Resin Strap Digital Sport Wat
2,Casio Men's PRW-3100YB-1CR Pro Trek Digital Displa,Casio Men's PRW-3100YB-1CR Pro Trek Digital Displa,Casio Black-Resin-10304195,Casio Black-Resin-10304195
3,Casio Women's Core LTP1351G-1C Gold Stainless-Stee,Casio 71604130 watchbands Genuine Factory Replacem,Casio G-5600E-1JF G-SHOCK Tough Solar Watch,CASIO G-SHOCK MULTIBAND 6 GXW-56BB-1JF Mens Japan
4,Casio 71604130 watchbands Genuine Factory Replacem,"Casio Men's Stainless Steel Quartz Resin Strap, Bl",Casio G-Shock Replacement Shell for DW5600E,Casio G-Shock GD100WW-7CS Whiteout Series All Whit
5,Casio G-Shock GD100WW-7CS Whiteout Series All Whit,FERENZI Women’s | Modern Silver-Tone and Red Large,CASIO G-Shock GA-2110SU-9AJF Japan Domestic,Casio Women's Core LTP1351G-1C Gold Stainless-Stee



🔎 Nike running shoes


,dense_v1,dense_v2,bm25,hybrid
1,Nike Kid's Flex 2015 RN Running Shoe (PS) Game Roy,Nike Little Kids Boys Revolution 2 Running Shoes-B,Nike Kid's Air Max 95 GS Running Shoes 7Y M US Bla,Nike Little Kids Boys Revolution 2 Running Shoes-B
2,Nike Little Kids Boys Revolution 2 Running Shoes-B,Nike Kid's Flex 2015 RN Running Shoe (PS) Game Roy,Nike Little Kids Boys Revolution 2 Running Shoes-B,Nike Kid's Air Max 95 GS Running Shoes 7Y M US Bla
3,Nike Kid's Air Max 95 GS Running Shoes 7Y M US Bla,Nike Kids Air Max Thea (GS) Wolf Grey/Hyper Pink/C,Nike Roshe G Jr Big Kid Golf Shoes 909250-004 Size,Nike Kid's Flex 2015 RN Running Shoe (PS) Game Roy
4,WHITEDREAM Women's Men's Comfortable Breathable Ru,Nike Kid's Air Max 95 GS Running Shoes 7Y M US Bla,Nike Kid's Flex 2015 RN Running Shoe (PS) Game Roy,Nike Kids Air Max Thea (GS) Wolf Grey/Hyper Pink/C
5,WHITEDREAM Women's Men's Comfortable Breathable Ru,COODO CD5002 Kids Athletic Sneakers Breathable Run,Nike Women's Dry Element Running Top Black Size Sm,Nike Women's Dry Element Running Top Black Size Sm



🔎 작게 나와서 한 치수 크게 사야 하는 청바지


,dense_v1,dense_v2,bm25,hybrid
1,1,Sunling Women's Sexy Skater Pleated Dresses Cute P,,Sunling Women's Sexy Skater Pleated Dresses Cute P
2,1,Romwe Women's Cute Mini Corduroy Button Down Pocke,,Romwe Women's Cute Mini Corduroy Button Down Pocke
3,1,NINEWE Women's 50S Retro Elegant Crossover One Pie,,NINEWE Women's 50S Retro Elegant Crossover One Pie
4,MTP01 SMALL BLUE,DIDK Women's Cat Print Cuffed Top and Polka Dots S,,DIDK Women's Cat Print Cuffed Top and Polka Dots S
5,Dani's Choice Cute Cartoon Character Socks (Sailor,Aenlley Womens Basic Shirts Stretchy Short Pleated,,Aenlley Womens Basic Shirts Stretchy Short Pleated


### 6-1. 직접 채점하기 (Precision@5)
검색에는 정답 라벨이 없어서, 실무에서도 사람이 직접 **"이 결과가 검색 의도에 맞나?"** 를 판단해 품질을 측정해요.

위 표를 보고 각 방법의 상위 5개 중 **관련 있는 상품 수(0~5)** 를 적어 주세요. 다 채우지 않아도 채운 것만 계산돼요.

In [ ]:
judgments = {
    '여름에 입기 좋은 시원한 린넨 원피스':     {'dense_v1': 0, 'dense_v2': 1, 'bm25': 0, 'hybrid': 1},
    '결혼식 하객룩 원피스':                   {'dense_v1': 1, 'dense_v2': 0, 'bm25': 0, 'hybrid': 0},
    '실버 원석 반지':                         {'dense_v1': 2, 'dense_v2': 0, 'bm25': 0, 'hybrid': 0},
    'Casio F91W':                             {'dense_v1': 4, 'dense_v2': 3, 'bm25': 3, 'hybrid': 4},
    '작게 나와서 한 치수 크게 사야 하는 청바지': {'dense_v1': 0, 'dense_v2': 0, 'bm25': 0, 'hybrid': 0},
}

p5 = pd.DataFrame(judgments).T.astype(float) / 5
print('Precision@5 (검색어별)')
display(p5.round(2))
print('평균')
display(p5.mean().round(2).to_frame('Precision@5').T)

Precision@5 (검색어별)


,dense_v1,dense_v2,bm25,hybrid
여름에 입기 좋은 시원한 린넨 원피스,0.0,0.2,0.0,0.2
결혼식 하객룩 원피스,0.2,0.0,0.0,0.0
실버 원석 반지,0.4,0.0,0.0,0.0
Casio F91W,0.8,0.6,0.6,0.8
작게 나와서 한 치수 크게 사야 하는 청바지,0.0,0.0,0.0,0.0


평균


,dense_v1,dense_v2,bm25,hybrid
Precision@5,0.28,0.16,0.12,0.2


In [ ]:
show_search('여름에 입기 좋은 시원한 린넨 원피스', k=5)
show_search('실버 원석 반지', k=5)

## 7. 콘텐츠 기반 추천으로 콜드 스타트 해결하기

02의 CF 모델은 학습에 없는 상품을 **절대** 추천할 수 없었어요 (정답의 19.5%).
임베딩은 구매 기록 없이 상품 내용만으로 만들어지니까, 신상품도 추천 후보가 될 수 있어요.

**방법**: 유저가 학습 기간에 산 상품들의 임베딩 평균 = 유저 취향 벡터 → 가장 가까운 상품 10개 추천 (이미 산 상품 제외)

**평가**: 02와 같은 정답으로 Recall@10을 계산
- 콜드 스타트 정답 유저 전체
- 전체 유저 중 무작위 2만 명 (02 결과와 비교용)

In [ ]:
asin2row = pd.Series(np.arange(len(items)), index=items['parent_asin'])
user_codes, user_index = pd.factorize(train_df['user_id'])
user2code = pd.Series(np.arange(len(user_index)), index=user_index)

# 유저 x 상품 행렬 (02와 같지만 열 순서가 items 순서)
U = sp.csr_matrix((np.ones(len(train_df), np.float32),
                   (user_codes, train_df['parent_asin'].map(asin2row).values)),
                  shape=(len(user_index), len(items)))
U.data[:] = 1.0

def recall_content(version, users_df, k=10, batch=4096):
    codes = user2code[users_df['user_id']].values
    targets = users_df['parent_asin'].map(asin2row).values
    hits = []
    for s in range(0, len(codes), batch):
        c = codes[s:s + batch]
        Q = np.ascontiguousarray(normalize(U[c] @ emb[version]), dtype=np.float32)
        n_seen = int(U[c].sum(axis=1).max())
        _, I = index[version].search(Q, k + min(n_seen, 200))
        for row, code, t in zip(I, c, targets[s:s + batch]):
            seen = set(U.indices[U.indptr[code]:U.indptr[code + 1]])
            recs = [j for j in row if j >= 0 and j not in seen][:k]
            hits.append(t in recs)
    return np.mean(hits)

cold_users = test_df[test_df['is_cold']]
sample_users = test_df.sample(min(20000, len(test_df)), random_state=0)

rows = []
for v in ['v1', 'v2']:
    rows.append({'model': f'Content ({v})',
                 'Recall@10 (콜드 스타트 정답)': recall_content(v, cold_users),
                 'Recall@10 (전체 샘플)': recall_content(v, sample_users)})
content_res = pd.DataFrame(rows).set_index('model')

# 02 결과 불러와서 같이 비교 (CF 모델은 콜드 스타트 정답을 구조적으로 못 맞힘 → 0)
cf = pd.read_csv(f'{MODEL_DIR}/rec_results_k{K}.csv', index_col=0)[['Recall@10']]
cf = cf.rename(columns={'Recall@10': 'Recall@10 (전체 샘플)'})
cf['Recall@10 (콜드 스타트 정답)'] = 0.0
compare = pd.concat([cf, content_res])[['Recall@10 (콜드 스타트 정답)', 'Recall@10 (전체 샘플)']]
print(f'콜드 스타트 정답 유저: {len(cold_users):,}명 | 전체 샘플: {len(sample_users):,}명')
compare.round(4)

콜드 스타트 정답 유저: 23,132명 | 전체 샘플: 20,000명


,Recall@10 (콜드 스타트 정답),Recall@10 (전체 샘플)
Popular,0.0000,0.0158
ItemCF,0.0000,0.0167
ALS,0.0000,0.0152
Content (v1),0.0244,0.0320
Content (v2),0.0226,0.0320


## 8. 02의 카시오 시계와 비교
02에서 ItemCF와 ALS는 카시오 F91W와 "비슷한 상품"으로 아동 블레이저, 양말, 신발끈 등을 추천했어요. 콘텐츠 기반은 어떨까요?

In [ ]:
with open(f'{MODEL_DIR}/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)
casio = maps['item_ids'][np.argsort(-maps['popularity'])[5]]   # 02의 8-1과 같은 상품
r = asin2row[casio]
show_items([r], '기준 상품')

for v in ['v1', 'v2']:
    scores, ids = index[v].search(emb[v][r:r + 1], 9)
    pairs = [(i, s) for i, s in zip(ids[0], scores[0]) if i != r][:8]
    show_items([i for i, _ in pairs], f'콘텐츠 기반 ({v})', [s for _, s in pairs])

## 9. 저장
4단계(리뷰 요약)와 5단계(RAG 챗봇)에서 이 인덱스를 그대로 불러와 씁니다.

In [ ]:
faiss.write_index(index['v2'], f'{EMB_DIR}/items_v2.faiss')
items[['parent_asin', 'title', 'store', 'price', 'average_rating', 'rating_number',
       'image_url', 'doc_v1', 'doc_v2']].to_parquet(f'{EMB_DIR}/items_index.parquet', index=False)
compare.to_csv(f'{MODEL_DIR}/content_vs_cf_k{K}.csv')
with open(f'{EMB_DIR}/config.pkl', 'wb') as f:
    pickle.dump({'model_name': MODEL_NAME, 'max_seq_length': 256,
                 'n_reviews': N_REVIEWS, 'query_prefix': 'query: ', 'passage_prefix': 'passage: '}, f)
print(sorted(os.listdir(EMB_DIR)))

['config.pkl', 'items_index.parquet', 'items_v1.npy', 'items_v2.faiss', 'items_v2.npy']


In [ ]:
print(f"리뷰가 붙은 상품: {(items['review_snippets'] != '').mean():.1%}")
wc = lambda s: s.str.split().str.len()
print(f"문서 길이 중앙값: v1 {wc(items['doc_v1']).median():.0f} → v2 {wc(items['doc_v2']).median():.0f}")

print('\n=== 7. 콘텐츠 기반 vs CF ===')
print(f'콜드 스타트 정답 유저: {len(cold_users):,}명')
print(compare.round(4).to_string())

print('\n=== 6. 검색 결과 비교 ===')
for q in QUERIES:
    print(f'\n🔎 {q}')
    print(compare_table(q).to_string())

리뷰가 붙은 상품: 96.8%
문서 길이 중앙값: v1 17 → v2 104

=== 7. 콘텐츠 기반 vs CF ===
콜드 스타트 정답 유저: 23,132명
              Recall@10 (콜드 스타트 정답)  Recall@10 (전체 샘플)
Popular                      0.0000             0.0158
ItemCF                       0.0000             0.0167
ALS                          0.0000             0.0152
Content (v1)                 0.0244             0.0320
Content (v2)                 0.0226             0.0320

=== 6. 검색 결과 비교 ===

🔎 여름에 입기 좋은 시원한 린넨 원피스
                                             dense_v1                                            dense_v2 bm25                                              hybrid
1  Womens Linen Overalls Shorts Summer Casual Rompers  Yuandy Womens Summer Casual Off Shoulder Print Jum       Yuandy Womens Summer Casual Off Shoulder Print Jum
2  Yuandy Womens Summer Casual Off Shoulder Print Jum  Mnemo Women's Summer Jumpsuits Sleeveless Casual P       Mnemo Women's Summer Jumpsuits Sleeveless Casual P
3  Soojun Women's Solid Cotton Linen Pants Sum

## 10. 정리해볼 질문 (블로그 / 포트폴리오용)

1. 리뷰를 섞은 v2는 v1보다 나았나? 어떤 검색어에서 차이가 컸고, 오히려 나빠진 경우도 있었나? (힌트: "작게 나와서 한 치수 크게" 같은 리뷰형 검색어)
2. BM25가 벡터 검색보다 잘한 검색어는? 한국어 검색어에서 BM25는 어땠나? → 하이브리드 검색이 필요한 이유
3. 직접 채점한 Precision@5에서 가장 좋은 방법은?
4. 콘텐츠 기반 추천은 콜드 스타트 정답을 몇 % 맞혔나? 전체 샘플에서는 02의 CF 모델과 비교해 어땠나? → 실제 서비스라면 두 방법을 어떻게 섞을까?
5. 카시오 시계의 '비슷한 상품'은 02의 ItemCF/ALS 결과와 어떻게 달랐나?


1. 리뷰를 섞은 v2는 v1보다 나았나?

상품의 96.8%에 리뷰가 붙었고, 문서 길이 중앙값은 17단어에서 104단어로 6배 늘었다. 하지만 검색 결과가 항상 좋아지지는 않았다.

v2가 나았던 경우: "출근할 때 입을 단정한 블라우스"에서 v1은 상위 5개 중 3개가 상품명이 비어 있는 상품이었고, v2는 벨 슬리브 블라우스 같은 관련 상품을 찾았다. 상품 정보가 거의 없는 상품도 리뷰 덕분에 검색할 수 있는 내용이 생겼다.

v2가 나빠진 경우: "실버 원석 반지"에서 v1은 자수정 원석 실버 상품을 찾았지만, v2는 구두, 스팀펑크 마스크, 체인 목걸이가 섞였다. "작게 나와서 한 치수 크게 사야 하는 청바지"에서는 v2가 사이즈 후기가 있는 원피스와 상의를 찾았지만 청바지는 하나도 없었다. 리뷰의 "작게 나온다"는 내용에 맞추느라 "청바지"라는 상품 종류를 놓친 것이다.

문서에 리뷰가 상품명보다 훨씬 길게 들어가면서, 상품이 무엇인지보다 리뷰에서 무슨 이야기를 했는지가 임베딩을 좌우하게 된 것으로 보인다. 개선 방향은 두 가지다. 리뷰를 그대로 넣지 않고 상품 속성만 요약해서 넣는 것(4단계 리뷰 요약), 그리고 검색어에서 "청바지"(상품 종류)와 "작게 나온다"(리뷰 속성)를 분리해서 상품 종류는 필터로 거르는 것(5단계 챗봇)이다.

검색 결과에서 발견한 문제 두 가지도 같이 적어 두면 좋아요.

상품명이 비어 있거나 “1"뿐인 상품 문제: v1에서는 여러 검색어의 상위에 이런 상품이 반복해서 나왔다. 내용이 거의 없는 문서는 임베딩이 특정 의미를 갖지 못하고 여러 검색어에 두루 가까워진다. 검색 대상에서 내용이 너무 짧은 문서를 빼는 전처리가 필요하다.

한국어와 영어의 의미 차이: "원피스"를 검색하면 드레스보다 점프수트와 롬퍼가 많이 나왔다. 한국어 원피스는 영어로 dress인데, 모델이 영어 “one piece”(위아래가 붙은 옷)로 이해한 것으로 보인다. "결혼식 하객룩"도 하객이 아니라 신부용 페티코트와 가운이 나왔다. 다국어 모델이 언어를 넘어 검색할 수는 있지만, 한국 쇼핑 표현의 미묘한 의미까지는 잡지 못했다.

이 두 가지는 면접에서 "실제로 결과를 들여다봤다"는 걸 보여주는 좋은 근거예요.

2. BM25가 벡터 검색보다 잘한 검색어는? 한국어 검색어에서는?

한국어 검색어 6개에서 BM25는 결과를 하나도 내지 못했다. 영어와 숫자만 인식하는 토큰화 방식이라 한국어 검색어와 영어 문서 사이에 겹치는 단어가 없기 때문이다. 그래서 한국어 검색어에서 하이브리드 결과는 벡터 검색(v2)과 완전히 같았다.

영어 검색어에서도 BM25가 벡터 검색보다 확실히 나은 경우는 없었다. “Casio F91W”는 네 방법 모두 1위에 정답 상품을 올렸다. 반대로 BM25의 약점은 분명했다. “Nike running shoes”에서 골프화와 러닝 상의가, “cozy pajamas for winter”에서 스카프와 모자 세트가 나왔다. 단어는 일치하지만 "신발", "잠옷"이라는 상품 종류를 이해하지 못한 것이다.

그래도 하이브리드 검색은 필요하다고 판단했다. 이번 검색어에는 모델 번호 검색이 1개뿐이라 BM25의 강점이 드러날 기회가 적었다. “GW300A”처럼 모델 번호만 입력하는 검색어를 더 넣어서 검증할 필요가 있다. 또 한국어 검색어를 LLM으로 영어로 바꿔서 BM25에도 넣으면, 한국어 검색에서도 하이브리드의 효과를 볼 수 있다.

3. 직접 채점한 Precision@5에서 가장 좋은 방법은?

제가 상품명만 보고 매긴 점수예요. 상품명이 50자에서 잘려서 애매한 상품이 있으니, 이미지로 확인하고 숫자를 조정한 뒤 노트북 6-1 셀에 붙여 넣으세요.

python
judgments = {
    '여름에 입기 좋은 시원한 린넨 원피스':     {'dense_v1': 0, 'dense_v2': 1, 'bm25': 0, 'hybrid': 1},
    '결혼식 하객룩 원피스':                   {'dense_v1': 1, 'dense_v2': 0, 'bm25': 0, 'hybrid': 0},
    '실버 원석 반지':                         {'dense_v1': 2, 'dense_v2': 0, 'bm25': 0, 'hybrid': 0},
    'Casio F91W':                             {'dense_v1': 4, 'dense_v2': 3, 'bm25': 3, 'hybrid': 4},
    '작게 나와서 한 치수 크게 사야 하는 청바지': {'dense_v1': 0, 'dense_v2': 0, 'bm25': 0, 'hybrid': 0},
}

채점 기준은 이렇게 잡았어요.

린넨 원피스: 드레스만 정답 (점프수트, 롬퍼는 오답)
하객룩: 하객이 입을 만한 파티 드레스만 정답
Casio F91W: 카시오 시계는 정답, 시계줄이나 부품은 오답

검색어 5개를 직접 채점한 결과, 평균 Precision@5는 dense_v1 0.28, dense_v2 0.16, BM25 0.12, 하이브리드 0.20으로 리뷰를 섞지 않은 v1이 가장 좋았다. 리뷰를 섞으면 검색이 좋아질 거라는 가설과 반대 결과다.

다만 검색어가 5개뿐이고 v1과 v2의 차이도 관련 상품 3개 정도라, 이 결과로 v1이 낫다고 결론 내리기는 어렵다. 또 혼자 상품명만 보고 채점해서 주관이 들어갔다. 실무에서는 검색어를 수십~수백 개로 늘리고 여러 사람이 채점하거나, 실제 검색 로그의 클릭 데이터로 평가한다.

5개 검색어 중 4개에서 모든 방법이 0~2점이었다. 의미 검색이 영어 상품명이 명확한 검색어(“Casio F91W”, 잠옷, 크로스백)에서는 잘 동작했지만, 한국어 쇼핑 표현과 리뷰 기반 조건이 섞인 검색어에서는 아직 부족했다.

4. 콘텐츠 기반 추천은 콜드 스타트 정답을 몇 % 맞혔나?

모델	콜드 스타트 정답 (23,132명)	전체 샘플
인기 추천	0%	1.58%
ItemCF	0%	1.67%
ALS	0%	1.52%
콘텐츠 (v1)	2.44%	3.20%
콘텐츠 (v2)	2.26%	3.20%

02의 CF 모델은 학습 데이터에 없는 상품을 구조적으로 추천할 수 없어서, 콜드 스타트 정답 유저 23,132명에게 Recall@10이 0%였다. 콘텐츠 기반 추천은 같은 유저들에게서 2.44%(v1)를 맞혔다. 구매 기록이 하나도 없는 신상품도 상품 정보만으로 추천할 수 있다는 것을 확인했다.

예상하지 못한 결과는 전체 유저에서도 콘텐츠 기반이 3.20%로, 가장 좋은 CF 모델인 ItemCF(1.67%)보다 약 1.9배 높았다는 점이다. 유저의 78%가 학습 기록이 1개뿐인 데이터라, 다른 유저의 구매 패턴에 의존하는 CF보다 "방금 산 상품과 비슷한 상품"을 찾는 방식이 더 잘 맞은 것으로 보인다.

리뷰를 섞은 v2는 추천에서도 v1보다 낫지 않았다. 콜드 스타트에서는 오히려 조금 낮았다(2.26%).

여기에는 꼭 확인할 점이 하나 있어요. 02에서 같은 상품이 다른 ID로 여러 번 등록된 경우를 봤고, 이번 검색 결과에서도 같은 상품이 3번씩 나왔잖아요. 콘텐츠 기반은 이런 사실상 같은 상품을 아주 잘 찾기 때문에, 점수 일부가 "같은 상품의 다른 색상이나 중복 등록"에서 나왔을 수 있어요. 그래서 이 한계도 같이 적어 두세요.

단, 적중한 경우 중 일부는 이미 산 상품과 사실상 같은 상품(다른 색상이나 중복 등록)일 수 있다. 이런 추천은 지표상 성공이지만 새로운 상품을 발견하게 해주지는 못한다. 적중 사례 중 중복 상품의 비율을 확인하는 것이 다음 과제다.

실제 서비스라면 두 방법을 섞어 쓴다. 판매 기록이 없는 신상품은 콘텐츠 기반으로만 추천 후보에 들어갈 수 있다. 기록이 적은 유저에게는 콘텐츠 기반의 비중을 높이고, 02에서 확인했듯 기록이 쌓인 유저에게는 ALS의 비중을 높인다. 두 모델의 추천 순위를 RRF로 합치는 방식을 다음 실험으로 해볼 수 있다.

5. 카시오 시계의 '비슷한 상품'은 02와 어떻게 달랐나?

같은 카시오 F91W 시계를 기준으로 비교했다.

02 ItemCF: 8개 중 시계 1개. 아동 블레이저, LED 신발 조명, 양말, 신발끈 등
02 ALS: 8개 중 시계 3개. 아기 부츠, 여성 원피스 등이 섞임
03 콘텐츠 v1: 8개 중 시계 7개, 카시오용 시계줄 1개
03 콘텐츠 v2: 8개 모두 시계, 그중 6개가 카시오 (G-Shock, Pro Trek 등)

구매 기록만 보는 CF는 소수의 유저가 우연히 함께 산 상품에 크게 흔들렸지만, 상품 내용을 보는 콘텐츠 기반은 "시계와 비슷한 상품"이라는 직관에 정확히 맞는 결과를 냈다. v2는 리뷰에 브랜드와 기능 이야기가 많아서 같은 브랜드의 디지털 스포츠 시계 쪽으로 더 모였다.

대신 콘텐츠 기반은 비슷한 상품만 찾고, 함께 필요한 상품은 찾지 못한다. 시계를 산 사람에게 시계를 8개 더 보여주는 것보다 시계줄이나 보호 케이스를 보여주는 게 나을 때도 있다. 그래서 상품 상세 페이지의 "비슷한 상품" 은 콘텐츠 기반으로, "함께 구매한 상품" 은 구매 기록이 충분한 상품에 한해 ItemCF로 나눠 쓰는 게 좋다.